# Model Pipeline

Run Bayesian models across different input scalings and compare DIC.

**Output structure:** `data/models/{csv_stem}/{version}/`

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from pyjags_pipeline import run_model, compare_dic, list_models

# Available input scalings
DATA_PATHS = {
    'per10k':             'data/wide_weekly_scaledPer10k.csv',
    'per-bed':            'data/wide_weekly_scaledPerBed.csv',
    'per-budget':         'data/wide_weekly_scaledPerBudgetThousands.csv',
    'per-65plus':         'data/wide_weekly_scaledPer1kOver65.csv',
}

print('Available models:', list_models())

Available models: ['v0.1', 'v1.1', 'v2.1', 'v2.2', 'v2.3', 'v2.4', 'v2.5', 'v2.6', 'v2.7', 'v2.8', 'v2.9', 'v3.1', 'v3.2', 'v3.3', 'v3.4', 'v3.5', 'v3.6']


## v4.x — Restructured development ladder (AR added last)

Mean structure is built first, autocorrelation is modelled last. The New Year effect is date-anchored from its introduction.

- v4.1 — Baseline (regional level `alpha[i]` only)
- v4.2 — + annual cycle (52-week cos/sin)
- v4.3 — + regional New Year, date-anchored (`delta_pm1, delta_pre, delta_mid, delta_post`)
- v4.4 — + Mid West reset (5-week block, `sigma_w1..sigma_w5`, gated to MW)
- v4.5 — + AR(1)
- **v4.6 — + AR(2)  [selected base]**

Extensions on the AR(2) base (each independent, all rejected by DIC):

- v4.7 — + 4-week cycle
- v4.8 — + 8-week cycle
- v4.9 — + 26-week cycle

v4.1–v4.4 omit AR, so their iid-Normal likelihood ignores week-to-week autocorrelation; their DIC is used for relative comparison of mean structure only.

In [ ]:
# Per-10k population: full v4.x development ladder (AR added last)
for version in ['v4.1', 'v4.2', 'v4.3', 'v4.4', 'v4.5', 'v4.6', 'v4.7', 'v4.8', 'v4.9']:
    result = run_model(
        version=version,
        data_path=DATA_PATHS['per10k'],
        seed=42,
    )
    print(f'{version}  DIC: {result.dic["DIC"]:.2f}  (deviance {result.dic["deviance"]:.2f}, pD {result.dic["penalty"]:.2f})')

# Reported model (v4.6, AR(2)) refit on the other response scalings used in the thesis
for scale in ['per-bed', 'per-budget', 'per-65plus']:
    result = run_model(
        version='v4.6',
        data_path=DATA_PATHS[scale],
        seed=42,
    )
    print(f'v4.6 [{scale}]  DIC: {result.dic["DIC"]:.2f}')

print('\nDIC comparison (per10k):')
compare_dic(DATA_PATHS['per10k'])[['version', 'description', 'DIC']]

## Regional-structure ablations (per-10k, DIC only)

Quantify how much the regional structure is worth by collapsing one piece to a single shared value and comparing DIC. Fit and exported for DIC only (no ranking/plots) because the global-level models have no per-region `alpha`.

Baseline-ladder ablations (no AR, no event terms):

- v4.10 — global level instead of `alpha[i]`; compared to v4.1, this isolates the value of regional levels.
- v4.11 — one shared cycle instead of `beta[i]`/`gamma[i]`; compared to v4.2, this isolates the value of a region-specific annual cycle.

Matched ablations of the reported model v4.6 (AR(2) + New Year + Mid West reset), each identical to v4.6 except the one regional change:

- v4.12 — v4.6 with a single global level; compared to v4.6, this isolates the regional level inside the full reported model.
- v4.13 — v4.6 with a single shared annual cycle; compared to v4.6, this isolates the region-specific cycle inside the full reported model.

In [ ]:
# Regional-structure ablations: fit + export DIC only (test_all needs per-region alpha)
from pyjags_pipeline import get_model

for version in ['v4.10', 'v4.11', 'v4.12', 'v4.13']:
    model = get_model(version)
    result = model.fit(data_path=DATA_PATHS['per10k'], seed=42)
    model.export(result)
    print(f'{version}  DIC: {result.dic["DIC"]:.1f}  (deviance {result.dic["deviance"]:.1f}, pD {result.dic["penalty"]:.1f})')

# Baseline-ladder deltas (lower DIC = regional structure justified)
print(compare_dic(DATA_PATHS['per10k'], versions=['v4.10', 'v4.1', 'v4.11', 'v4.2']).to_string(index=False))
# Matched-to-v4.6 deltas (apples-to-apples against the reported model)
print(compare_dic(DATA_PATHS['per10k'], versions=['v4.12', 'v4.13', 'v4.6']).to_string(index=False))

## Annual cycle: amplitude/phase parameterisation (per-10k)

Two variants of the reported AR(2) model. The annual cycle is written directly as `amp * sin(2*pi*t/52 + phase)` instead of the `cos`/`sin` pair, so amplitude and phase are sampled as parameters rather than reconstructed post-hoc.

- v4.14 — region-specific amplitude `amp[i]` and region-specific phase `phase[i]`.
- v4.15 — region-specific amplitude `amp[i]`, single global phase `phase`.

Everything else matches the reported model: date-anchored regional New Year, Mid West reset, AR(2). `amp` is constrained positive so amplitude/phase are identified. DIC is comparable to the reported model because the likelihood is unchanged.

In [ ]:
# Annual cycle amplitude/phase variants, per-10k
for version in ['v4.14', 'v4.15']:
    result = run_model(
        version=version,
        data_path=DATA_PATHS['per10k'],
        seed=42,
    )
    print(f'{version}  DIC: {result.dic["DIC"]:.2f}  (deviance {result.dic["deviance"]:.2f}, pD {result.dic["penalty"]:.2f})')

# DIC vs the reported cos/sin model (v4.6)
print(compare_dic(DATA_PATHS['per10k'], versions=['v4.6', 'v4.14', 'v4.15']).to_string(index=False))